In [ ]:
import torch

# Check GPU availability
print("="*60)
print("GPU CHECK")
print("="*60)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"\n✅ GPU is available and will be used!")
    print(f"Current GPU: {torch.cuda.current_device()}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / (1024**3):.1f} GB")
else:
    print(f"\n⚠️ GPU NOT available - will use CPU (slower)")


GPU CHECK
CUDA available: False
CUDA device: N/A
Device count: 0

⚠️ GPU NOT available - will use CPU (slower)


In [ ]:
import torch
from pathlib import Path
import glob
from training.dataloader import DACDataset
from torch.utils.data import DataLoader

# Setup paths for DAC embeddings
root = Path(r"c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift")
data_dir = root / "data" / "output"

non_rock_dir = data_dir / "non_rock_dac"
rock_dir = data_dir / "rock_dac"

# Check if DAC directories exist
print(f"Non-rock DAC dir exists: {non_rock_dir.exists()}")
print(f"Rock DAC dir exists: {rock_dir.exists()}")

# Get file lists
source_files = sorted(glob.glob(str(non_rock_dir / "*.pt")))[:8]  # Use 8 files for testing
target_files = sorted(glob.glob(str(rock_dir / "*.pt")))[:8]

print(f"\nSource files: {len(source_files)}")
print(f"Target files: {len(target_files)}")

if not source_files or not target_files:
    print("\n⚠️ No DAC files found! Run audio-to-DAC conversion first.")
    print("Use NeuralCodecConverter.save_codec() to convert audio to DAC embeddings")
else:
    # Create dataset
    dataset = DACDataset(source_files, target_files)
    print(f"\nDataset size: {len(dataset)}")

    # Test single sample
    x0, x1 = dataset[0]
    print(f"\nSingle sample shapes (should be 2D: T, D):")
    print(f"  x0: {x0.shape}")
    print(f"  x1: {x1.shape}")
    print(f"  Latent dim: {x0.shape[-1]} (should be 768 for DAC)")

    from training.training import TrainingPipeline, TrainingConfig

    # Create config for DAC embeddings
    config = TrainingConfig(
        num_epochs=1,
        batch_size=1,
        learning_rate=1e-4,
        checkpoint_interval=1,
        max_time=512,           # max time frames
        input_dim=768,          # DAC latent dimension
        embed_dim=128,
        num_blocks=2,
        num_heads=2,
        hidden_dim=512,
        dropout=0.1,
        num_genres=2,
    )

    pipeline = TrainingPipeline(config)

    # Use pipeline's data setup
    loader = pipeline.setup_data(source_files, target_files)

    # Test batch loading
    batch = next(iter(loader))
    x0_batch, x1_batch, mask_batch = batch

    print(f"\nBatch shapes (should be 3D: B, T, D):")
    print(f"  x0_batch: {x0_batch.shape}")
    print(f"  x1_batch: {x1_batch.shape}")
    print(f"  mask: {mask_batch.shape}")

    # Verify 3D
    if x0_batch.ndim == 3 and x1_batch.ndim == 3:
        print(f"\n✅ SUCCESS: Tensors are 3D for DAC!")
        print(f"   Batch size: {x0_batch.shape[0]}")
        print(f"   Time steps: {x0_batch.shape[1]}")
        print(f"   Latent dim: {x0_batch.shape[2]}")
    else:
        print(f"\n❌ ERROR: Expected 3D tensors for DAC!")

Source files: 8
Target files: 8

Dataset size: 8

Single sample shapes:
  x0: torch.Size([1, 100, 2811]) (should be 3D: C, H, W)
  x1: torch.Size([1, 100, 2811])
film_conditioner.py STARTED

Batch shapes (should be 4D: B, C, H, W):
  x0_batch: torch.Size([1, 1, 100, 1024])
  x1_batch: torch.Size([1, 1, 100, 1024])
  mask: torch.Size([1, 1, 1, 1024])

✅ SUCCESS: Tensors are 4D!
   Batch size: 1
   Channels: 1
   Mel bins: 100
   Time steps: 1024


In [ ]:
from training.training import TrainingConfig, TrainingPipeline

# Create config for DAC embeddings (lower memory for testing)
config = TrainingConfig(
    num_epochs=1,
    batch_size=1,
    learning_rate=1e-4,
    checkpoint_interval=1,
    max_time=512,           # max time frames for DAC
    input_dim=768,          # DAC latent dimension
    embed_dim=128,
    num_blocks=2,
    num_heads=2,
    hidden_dim=512,
    dropout=0.1,
    num_genres=2,
)

print(f"✓ Config created for DAC embeddings:")
print(f"  Device: {config.device}")
print(f"  Input dim: {config.input_dim} (DAC latent)")
print(f"  Embed dim: {config.embed_dim}")
print(f"  Epochs: {config.num_epochs}")
print(f"  Batch size: {config.batch_size}")

# Step 1: Import & Setup Training Pipeline
print("\n" + "="*60)
print("STEP 1: Import & Setup Training Pipeline")
print("="*60)

✓ Config created:
  Device: cpu
  Epochs: 1
  Batch size: 1

STEP 1: Import & Setup Training Pipeline


# Step 2: Initialize Pipeline
**What this does:** Creates the DiT model, FlowMatching wrapper, optimizer, and scheduler. This happens in the __init__ method.

In [4]:
import time

print("Initializing pipeline (this takes ~5-10 seconds)...")
start = time.time()

pipeline = TrainingPipeline(config)

init_time = time.time() - start
print(f"✓ Pipeline initialized in {init_time:.1f}s")
print(f"  DiT model: {sum(p.numel() for p in pipeline.dit.parameters())} parameters")
print(f"  Device: {pipeline.device}")
print(f"  Optimizer: {type(pipeline.optimizer).__name__}")
print(f"  Scheduler: {type(pipeline.scheduler).__name__}")

Initializing pipeline (this takes ~5-10 seconds)...
✓ Pipeline initialized in 0.0s
  DiT model: 1488928 parameters
  Device: cpu
  Optimizer: AdamW
  Scheduler: ReduceLROnPlateau


# Step 3: Setup Data
**What this does:** Creates a DACDataset and wraps it in a DataLoader with a custom collate_fn that handles padding for variable-length DAC embeddings.

In [5]:
print("\n" + "="*60)
print("STEP 3: Setup Data (Using Existing Variables)")
print("="*60)

print(f"\nUsing already loaded files:")
print(f"  Source files: {len(source_files)}")
print(f"  Target files: {len(target_files)}")

# Setup loader using existing source_files and target_files
loader = pipeline.setup_data(source_files, target_files)

print(f"✓ DataLoader created:")
print(f"  Dataset size: {len(loader.dataset)}")
print(f"  Number of batches: {len(loader)}")
print(f"  Batch size: {config.batch_size}")


STEP 3: Setup Data (Using Existing Variables)

Using already loaded files:
  Source files: 8
  Target files: 8
✓ DataLoader created:
  Dataset size: 8
  Number of batches: 8
  Batch size: 1


# Step 4: Load One Batch
**What this does:** Gets the first batch from the loader. The collate_fn pads all DAC embeddings to the same length and creates masks.

In [ ]:
print("\n" + "="*60)
print("STEP 4: Load One Batch (DAC Embeddings)")
print("="*60)

print("\nLoading batch from DataLoader...")

# Get fresh batch from existing loader
batch = next(iter(loader))
x0_batch, x1_batch, mask_batch = batch

print(f"✓ Batch loaded (DAC format):")
print(f"  x0 (source):  {x0_batch.shape}  (batch, time, latent_dim)")
print(f"  x1 (target):  {x1_batch.shape}  (batch, time, latent_dim)")
print(f"  mask:         {mask_batch.shape}  (batch, time)")
print(f"  Valid frames: {mask_batch.sum().item():.0f}")
print(f"  Padded frames: {(1 - mask_batch).sum().item():.0f}")


STEP 4: Load One Batch (Using Existing DataLoader)

Using already loaded batch from loader...
✓ Batch loaded:
  x0 (source):  torch.Size([1, 1, 100, 1024])  (batch, channels, mels, time)
  x1 (target):  torch.Size([1, 1, 100, 1024])  (batch, channels, mels, time)
  mask:         torch.Size([1, 1, 1, 1024])  (batch, channels, mels, time)
  Valid frames: 1024
  Padded frames: 0


# Step 5: Test Forward Pass (No Training)
**What this does:** 
1. Moves batch to device (GPU/CPU)
2. Applies mask to ignore padded regions (expands mask for 3D DAC tensors)
3. Calls flow.compute_loss() which:
   - Samples random timesteps t
   - Interpolates between x0 and x1: x(t) = (1-t)*x0 + t*x1
   - Predicts velocity from DiT
   - Computes MSE loss between predicted and true velocity
4. Shows loss value (should be random/high at first)

In [ ]:
print("\n" + "="*60)
print("STEP 5: Test Forward Pass (No Training)")
print("="*60)

print("\nTesting forward pass (no gradients)...\n")

# Move to device
x0 = x0_batch.to(pipeline.device)
x1 = x1_batch.to(pipeline.device)
mask = mask_batch.to(pipeline.device)

# Apply mask to ignore padding
# mask is (B, T), expand to (B, T, 1) for broadcasting with (B, T, D)
mask_expanded = mask.unsqueeze(-1)
x0 = x0 * mask_expanded
x1 = x1 * mask_expanded

# Set genre (1 = rock)
genre_ids = torch.ones(x0.size(0), device=pipeline.device, dtype=torch.long)

print(f"Input shapes after masking:")
print(f"  x0: {x0.shape} (B, T, D)")
print(f"  x1: {x1.shape} (B, T, D)")
print(f"  genre_ids: {genre_ids.shape}")

# Compute loss (no backward yet)
with torch.no_grad():
    loss = pipeline.flow.compute_loss(x0, x1, genre_ids)

print(f"\n✓ Forward pass successful!")
print(f"  Loss: {loss.item():.4f}")
print(f"  (High loss is expected - model hasn't learned anything yet)")


STEP 5: Test Forward Pass (No Training)

Testing forward pass (no gradients)...

Input shapes after masking:
  x0: torch.Size([1, 1, 100, 1024])
  x1: torch.Size([1, 1, 100, 1024])
  genre_ids: torch.Size([1])

✓ Forward pass successful!
  Loss: 40821.1562
  (High loss is expected - model hasn't learned anything yet)


# Step 6: One Training Step
**What this does:**
1. Computes loss (with gradients enabled)
2. Backward pass: calculates gradients
3. Clips gradients (prevents exploding gradients)
4. Optimizer step: updates model weights using gradient descent
5. Shows how loss changes with one update

In [8]:
print("\n" + "="*60)
print("STEP 6: One Training Step")
print("="*60)

print("\nRunning ONE training step...\n")

# Switch to training mode
pipeline.flow.train()

# Compute loss
loss = pipeline.flow.compute_loss(x0, x1, genre_ids)
print(f"1. Loss computed: {loss.item():.4f}")

# Backward pass: compute gradients
pipeline.optimizer.zero_grad()  # Clear old gradients
loss.backward()                  # Compute new gradients
print(f"2. Gradients computed")

# Check gradient norms (sanity check)
total_grad_norm = 0
for p in pipeline.flow.parameters():
    if p.grad is not None:
        total_grad_norm += p.grad.norm().item() ** 2
total_grad_norm = total_grad_norm ** 0.5
print(f"   Gradient norm: {total_grad_norm:.4f}")

# Clip gradients (prevents exploding gradients)
torch.nn.utils.clip_grad_norm_(pipeline.flow.parameters(), config.grad_clip_norm)
print(f"3. Gradients clipped (max norm: {config.grad_clip_norm})")

# Update weights
pipeline.optimizer.step()
print(f"4. Weights updated by optimizer")
print(f"   Learning rate: {pipeline.optimizer.param_groups[0]['lr']}")

# Compute loss again to see if it changed
with torch.no_grad():
    new_loss = pipeline.flow.compute_loss(x0, x1, genre_ids)

print(f"\n✓ Training step complete!")
print(f"  Before step: {loss.item():.4f}")
print(f"  After step:  {new_loss.item():.4f}")
print(f"  Change: {(new_loss.item() - loss.item()):.4f}")


STEP 6: One Training Step

Running ONE training step...

1. Loss computed: 40821.0664
2. Gradients computed
   Gradient norm: 81.0834
3. Gradients clipped (max norm: 1.0)
4. Weights updated by optimizer
   Learning rate: 0.0001

✓ Training step complete!
  Before step: 40821.0664
  After step:  40819.9180
  Change: -1.1484


# Step 7: Full Training Loop (1 Epoch)
**What this does:** Runs the train() method for 1 epoch with 1 batch, showing loss history.

In [9]:
print("\n" + "="*60)
print("STEP 7: Full Training Loop (1 Epoch)")
print("="*60)

print("\nRunning full training loop (1 epoch)...\n")
print("=" * 60)

losses = pipeline.train(loader)

print("=" * 60)
print(f"\n✓ Training complete!")
print(f"  Losses: {losses}")
print(f"  Final loss: {losses[-1]:.4f}")


STEP 7: Full Training Loop (1 Epoch)

Running full training loop (1 epoch)...


Epoch 1/1 started
Starting batch shapes:
  x0: torch.Size([1, 1, 100, 1024])
  x1: torch.Size([1, 1, 100, 1024])
  mask: torch.Size([1, 1, 1, 1024])
Epoch 1/1 - Step 8/8 - Loss: 141161.0938
Epoch 1/1 done - Avg Loss: 431335.6865
Saving checkpoint for epoch 1...
✓ Saved best model: loss=431335.6865
Checkpoint saved for epoch 1: checkpoints\checkpoint_epoch_001.pt

✓ Training complete!
  Losses: [431335.6865234375]
  Final loss: 431335.6865


# Step 8: Check Saved Checkpoint
**What this does:** Lists the saved checkpoints and verifies the best_model.pt was created.

In [10]:
print("\n" + "="*60)
print("STEP 8: Check Saved Checkpoint")
print("="*60)

import os

checkpoint_dir = Path("checkpoints")
if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob("*.pt"))
    print(f"\n✓ Found {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        size_mb = os.path.getsize(ckpt) / (1024 * 1024)
        print(f"  {ckpt.name} ({size_mb:.1f} MB)")
        
        # Load and show info
        state = torch.load(ckpt, map_location='cpu')
        print(f"    - Epoch: {state['epoch']}, Loss: {state['avg_loss']:.4f}")
else:
    print("\n❌ No checkpoints found")

print("\n" + "="*60)
print("TRAINING TEST COMPLETE!")
print("="*60)


STEP 8: Check Saved Checkpoint

✓ Found 2 checkpoint(s):
  best_model.pt (17.1 MB)
    - Epoch: 1, Loss: 431335.6865
  checkpoint_epoch_001.pt (17.1 MB)
    - Epoch: 1, Loss: 431335.6865

TRAINING TEST COMPLETE!
